# 01 · Explore the Hugging Face data

Goal: get a feeling for the data before building features.

WSL has only ~7 GB RAM. July 2024 (2 M rows) fits easily. For the big months from 2025-11 on (~15 M rows), select columns and filter **before** `.collect()`.

In [ ]:
from pathlib import Path

import polars as pl

DATA_DIR = Path("../data/raw/hf/monthly_processed_data")
sorted(p.name for p in DATA_DIR.glob("*.parquet"))

In [ ]:
# scan = lazy: nothing is loaded yet, Polars only reads the file's schema
lf = pl.scan_parquet(DATA_DIR / "data-2024-07.parquet") # data-2026-06.parquet
lf.collect_schema()

In [ ]:
df = lf.collect()  # now the data is loaded into memory
print(df.shape)
df.head()

In [ ]:
# Arrival delay computed by ourselves (delay_in_min is mostly the departure)
arrivals = df.filter(
    pl.col("arrival_planned_time").is_not_null(),
    ~pl.col("arrival_is_canceled"),
).with_columns(
    arrival_delay_min=(pl.col("arrival_change_time") - pl.col("arrival_planned_time")).dt.total_minutes(),
).with_columns(
    y=(pl.col("arrival_delay_min") >= 6).cast(pl.Int8),
)
arrivals.select("station_name", "train_type", "train_number", "arrival_planned_time", "arrival_delay_min", "y").head(10)

In [ ]:
# Share of late arrivals per train type
arrivals.group_by("train_type").agg(
    pl.len().alias("stops"),
    pl.col("y").mean().round(3).alias("late_share"),
).sort("stops", descending=True).head(15)

## Questions to explore yourself

1. What does the distribution of `arrival_delay_min` look like? (Tip: `arrivals["arrival_delay_min"].describe()`, then a histogram)
2. At which hour of the day is the late share highest? (Tip: `pl.col("arrival_planned_time").dt.hour()`)
3. Which 10 stations have the highest late share?
4. How many stops have exactly 0 minutes delay? Is that plausible?
5. Pick one `train_line_ride_id` and look at all its stops sorted by `train_line_station_num`. How does the delay change along the ride?
6. Compare July 2024 with September 2026: did the late share change?